# Optimización de rutas OSMR y exportación a Excel
Este notebook replica el algoritmo de optimización por OSMR, pero solo genera el archivo Excel bajo los mismos parámetros de selección y optimización.

In [6]:
# 1. Importar librerías necesarias
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import networkx as nx
import requests
import openpyxl
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
import os
from datetime import datetime

In [7]:
# 2. Cargar datos desde Excel (ahora desde carpeta Input)
input_folder = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Input'
archivos = [f for f in os.listdir(input_folder) if f.lower().endswith(('.xlsx', '.xls'))]
if archivos:
    ARCHIVO_DATOS = os.path.join(input_folder, archivos[0])
    print(f'Archivo de entrada detectado: {ARCHIVO_DATOS}')
else:
    raise FileNotFoundError('No se encontró ningún archivo Excel en la carpeta Input')
df = pd.read_excel(ARCHIVO_DATOS, sheet_name=None)
df_clientes = df[list(df.keys())[0]]
df_cdr = df[list(df.keys())[1]]

Archivo de entrada detectado: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Input\Clientes_Ruta_01072025.xlsx


In [ ]:
# 3. Configurar parámetros de optimización (widgets)
rutas_disponibles = sorted(df_clientes['Ruta'].dropna().unique())
clientes_por_ruta = {ruta: df_clientes[df_clientes['Ruta'] == ruta].shape[0] for ruta in rutas_disponibles}
rutas_labels = [f"{ruta} ({clientes_por_ruta[ruta]} clientes)" for ruta in rutas_disponibles]
rutas_widget = widgets.SelectMultiple(options=list(zip(rutas_labels, rutas_disponibles)), value=tuple(rutas_disponibles), description='Rutas a optimizar:', style={'description_width': 'initial'}, layout=widgets.Layout(width='60%'))
mets_disponibles = sorted(df_cdr['CodMET'].dropna().unique())
met_checkboxes = [widgets.Checkbox(value=False, description=str(met), indent=False) for met in mets_disponibles]
met_box = widgets.VBox([widgets.Label('Selecciona uno o varios METs:')] + met_checkboxes)
clientes_a_procesar = widgets.IntText(value=10, description='Cantidad de clientes a procesar:', style={'description_width': 'initial'}, layout=widgets.Layout(width='30%'))
todos_clientes_checkbox = widgets.Checkbox(value=False, description='Procesar todos los clientes', indent=False)
select_all_btn = widgets.Button(description='Seleccionar todo', button_style='info')
deselect_all_btn = widgets.Button(description='Deseleccionar todo', button_style='warning')
param_button = widgets.Button(description='Aplicar selección', button_style='success')
output_param = widgets.Output()
def seleccionar_todo(b): rutas_widget.value = tuple(rutas_disponibles)
def deseleccionar_todo(b): rutas_widget.value = tuple()
def aplicar_parametros(b):
    with output_param:
        clear_output()
        mets_seleccionados = [met.description for met in met_checkboxes if met.value]
        print(f'MET(s) seleccionados: {mets_seleccionados}')
        rutas_seleccionadas = list(rutas_widget.value)
        print(f'Rutas a optimizar: {rutas_seleccionadas}')
        if todos_clientes_checkbox.value:
            print('Procesando el 100% de los clientes de las rutas seleccionadas.')
        else:
            print(f'Cantidad de clientes a procesar: {clientes_a_procesar.value}')
select_all_btn.on_click(seleccionar_todo)
deselect_all_btn.on_click(deseleccionar_todo)
param_button.on_click(aplicar_parametros)
display(widgets.VBox([met_box, rutas_widget, widgets.HBox([clientes_a_procesar, todos_clientes_checkbox]), widgets.HBox([select_all_btn, deselect_all_btn]), param_button, output_param]))
# Los parámetros seleccionados estarán en met_checkboxes, rutas_widget.value, clientes_a_procesar.value y todos_clientes_checkbox.value para usarse en la optimización.

In [9]:
# 4. Obtener matriz de distancias con OSRM
def obtener_matriz_distancias_osrm(puntos):
    n = len(puntos)
    matriz = [[0]*n for _ in range(n)]
    url_base = 'http://router.project-osrm.org/route/v1/driving/'
    session = requests.Session()  # Usar sesión para acelerar múltiples requests
    for i in range(n):
        for j in range(n):
            if i == j:
                matriz[i][j] = 0
            else:
                coords = f"{puntos[i][0]},{puntos[i][1]};{puntos[j][0]},{puntos[j][1]}"
                url = url_base + coords + '?overview=false'
                try:
                    r = session.get(url, timeout=5)
                    data = r.json()
                    matriz[i][j] = data['routes'][0]['distance']/1000 if 'routes' in data and data['routes'] else float('inf')
                except:
                    matriz[i][j] = float('inf')
    return matriz

In [10]:
# 5 y 6. Resolver TSP para rutas seleccionadas y exportar a Excel con formato profesional
def resolver_tsp(matriz):
    n = len(matriz)
    G = nx.Graph()
    for i in range(n):
        for j in range(n):
            if i != j:
                G.add_edge(i, j, weight=matriz[i][j])
    ciclo = nx.approximation.traveling_salesman_problem(G, weight='weight', cycle=True)
    distancia_total = sum(matriz[ciclo[i]][ciclo[i+1]] for i in range(len(ciclo)-1))
    return ciclo, distancia_total

# Procesar parámetros seleccionados y preparar datos para Excel
output_dir = r'C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs'
os.makedirs(output_dir, exist_ok=True)
fecha_hora = datetime.now().strftime('%Y%m%d_%H%M%S')
mets_seleccionados = [met.description for met in met_checkboxes if met.value]
rutas_seleccionadas = list(rutas_widget.value)
num_clientes = clientes_a_procesar.value
export_rows = []
resumen_rutas = []
total_clientes = 0
distancia_total = 0
max_dist_general = 0
max_from_general = ''
max_to_general = ''
for met in mets_seleccionados:
    met_row = df_cdr[df_cdr['CodMET'] == met].iloc[0]
    for ruta in rutas_seleccionadas:
        if todos_clientes_checkbox.value:
            clientes_ruta = df_clientes[df_clientes['Ruta'] == ruta]
        else:
            clientes_ruta = df_clientes[df_clientes['Ruta'] == ruta].head(num_clientes)
        if clientes_ruta.empty:
            continue
        puntos = [(met_row['U_longitud'], met_row['U_latitud'])] + [(row['U_longitud'], row['U_latitud']) for _, row in clientes_ruta.iterrows()]
        codigos = [met] + list(clientes_ruta['CodCli'])
        matriz = obtener_matriz_distancias_osrm(puntos)
        ciclo, distancia_total_ruta = resolver_tsp(matriz)
        ciclo_filtrado = []
        vistos = set()
        for idx, val in enumerate(ciclo):
            if idx == 0 or idx == len(ciclo)-1 or val not in vistos:
                ciclo_filtrado.append(val)
                vistos.add(val)
        if ciclo_filtrado[-1] != 0:
            ciclo_filtrado.append(0)
        recorrido_codigos = [codigos[i] for i in ciclo_filtrado]
        distancias = []
        max_dist = 0
        max_from = ''
        max_to = ''
        for i in range(len(ciclo_filtrado)-1):
            d = matriz[ciclo_filtrado[i]][ciclo_filtrado[i+1]]
            distancias.append(d)
            if d > max_dist:
                max_dist = d
                max_from = codigos[ciclo_filtrado[i]]
                max_to = codigos[ciclo_filtrado[i+1]]
        total_clientes_ruta = len(clientes_ruta)
        distancia_promedio_ruta = distancia_total_ruta / total_clientes_ruta if total_clientes_ruta > 0 else 0
        resumen_rutas.append({
            'MET': met,
            'Ruta': ruta,
            'Clientes': total_clientes_ruta,
            'Distancia_total_km': distancia_total_ruta,
            'Distancia_promedio_cliente_km': distancia_promedio_ruta,
            'Distancia_maxima_km': max_dist,
            'De': max_from,
            'A': max_to
        })
        total_clientes += total_clientes_ruta
        distancia_total += distancia_total_ruta
        if max_dist > max_dist_general:
            max_dist_general = max_dist
            max_from_general = max_from
            max_to_general = max_to
        for idx, sec in enumerate(ciclo_filtrado):
            distancia_val = distancias[idx] if idx < len(distancias) else ''
            if sec == 0:
                export_rows.append({
                    'MET': met,
                    'Ruta': ruta,
                    'Secuencia': idx+1,
                    'Tipo': 'MET',
                    'Codigo': met,
                    'Longitud': met_row['U_longitud'],
                    'Latitud': met_row['U_latitud'],
                    'Nombre': met_row.get('Nombre', ''),
                    'Distancia_al_siguiente_km': distancia_val,
                    'Recorrido': ' -> '.join([str(x) for x in recorrido_codigos]),
                    'Distancia_total_km': distancia_total_ruta
                })
            else:
                cli_row = clientes_ruta.iloc[sec-1]
                export_rows.append({
                    'MET': met,
                    'Ruta': ruta,
                    'Secuencia': idx+1,
                    'Tipo': 'Cliente',
                    'Codigo': cli_row['CodCli'],
                    'Longitud': cli_row['U_longitud'],
                    'Latitud': cli_row['U_latitud'],
                    'Nombre': cli_row.get('Nombre', ''),
                    'Distancia_al_siguiente_km': distancia_val,
                    'Recorrido': ' -> '.join([str(x) for x in recorrido_codigos]),
                    'Distancia_total_km': distancia_total_ruta
                })
# Exportar a Excel con formato profesional
df_export = pd.DataFrame(export_rows)
df_resumen = pd.DataFrame(resumen_rutas)
resumen_general = pd.DataFrame([{
    'Total_clientes': total_clientes,
    'Distancia_total_km': distancia_total,
    'Distancia_promedio_cliente_km': distancia_total / total_clientes if total_clientes > 0 else 0,
    'Distancia_maxima_km': max_dist_general,
    'De': max_from_general,
    'A': max_to_general
}])
excel_path = os.path.join(output_dir, f'recorridos_optimos_todos_{fecha_hora}.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_export.to_excel(writer, sheet_name='Recorridos', index=False)
    df_resumen.to_excel(writer, sheet_name='Resumen por ruta', index=False)
    resumen_general.to_excel(writer, sheet_name='Resumen general', index=False)
wb = openpyxl.load_workbook(excel_path)
header_font = Font(bold=True, color='FFFFFF', size=12)
header_fill = PatternFill('solid', fgColor='4F81BD')
cell_fill = PatternFill('solid', fgColor='DDEBF7')
border = Border(left=Side(style='thin', color='4F81BD'), right=Side(style='thin', color='4F81BD'), top=Side(style='thin', color='4F81BD'), bottom=Side(style='thin', color='4F81BD'))
align = Alignment(horizontal='center', vertical='center', wrap_text=True)
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    for col_idx, cell in enumerate(ws[1], 1):
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = align
        cell.border = border
        if 'Ruta' in cell.value: cell.value = '🛣️ ' + str(cell.value)
        if 'Cliente' in cell.value: cell.value = '👨‍💼 ' + str(cell.value)
        if 'Distancia' in cell.value: cell.value = '📏 ' + str(cell.value)
        if 'Secuencia' in cell.value: cell.value = '🔢 ' + str(cell.value)
        if 'MET' in cell.value: cell.value = '🏠 ' + str(cell.value)
        if 'Nombre' in cell.value: cell.value = '📚 ' + str(cell.value)
        if 'De' == cell.value: cell.value = '🔴 De'
        if 'A' == cell.value: cell.value = '🟢 A'
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.fill = cell_fill
            cell.alignment = align
            cell.border = border
    for col in ws.columns:
        max_length = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                if cell.value:
                    max_length = max(max_length, len(str(cell.value)))
            except:
                pass
        ws.column_dimensions[col_letter].width = max(12, min(max_length + 2, 40))
    if sheet_name in ['Resumen general', 'Resumen por ruta']:
        for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
            for cell in row:
                cell.fill = PatternFill('solid', fgColor='FFD966')
wb.save(excel_path)
print(f'Excel grupal exportado a: {excel_path}')

Excel grupal exportado a: C:\Users\igcamposg\OneDrive - Truper, S.A. de C.V\Documentos\Proyecto\Celaya\Outputs\recorridos_optimos_todos_20260211_123720.xlsx
